# Results — FID vs. Sampling Steps

Reproduces every figure used in the report: trains Flow Matching and DDPM,
(optionally) a reflowed model, then evaluates FID as a function of the number of
sampling steps. Run top to bottom on a Kaggle T4.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import os
if not os.path.isdir('flow-matching-fewstep'):
    !git clone -q https://github.com/xHansi/flow-matching-fewstep.git
REPO = os.path.abspath('flow-matching-fewstep')
!pip install -q pytorch-fid
print('repo at', REPO)

## 1. Train both methods

Bump `--epochs` for final results (e.g. 30). Each run saves a checkpoint, a loss
curve and a sample grid under `results/<method>_mnist/`.

In [ ]:
!cd {REPO} && PYTHONPATH={REPO} python -m scripts.train --method flow --dataset mnist --epochs 30

In [ ]:
!cd {REPO} && PYTHONPATH={REPO} python -m scripts.train --method ddpm --dataset mnist --epochs 30

## 2. (Optional) Reflow the Flow Matching model

Straightens the ODE paths using the model's own noise->sample pairs.

In [ ]:
!cd {REPO} && PYTHONPATH={REPO} python -m scripts.reflow --ckpt results/flow_mnist/ckpt.pt --pairs 50000 --gen-steps 100 --epochs 30

## 3. Loss curves

In [ ]:
from IPython.display import Image, display
for m in ('flow', 'ddpm'):
    print(m); display(Image(f'{REPO}/results/{m}_mnist/loss_curve.png'))

## 4. FID vs. steps

Measured **without** classifier-free guidance (`--cfg-scale 1.0`): guidance
inflates FID and over-saturates DDPM at high step counts, so it would confound
the steps comparison. The qualitative grids below use guidance for readability.
DDPM uses DDIM here (`--eta 0`); pass `--eta 1` for ancestral DDPM.

In [ ]:
!cd {REPO} && PYTHONPATH={REPO} python -m scripts.eval_fid \
    --ckpts flow=results/flow_mnist/ckpt.pt ddpm=results/ddpm_mnist/ckpt.pt reflow=results/reflow_mnist/ckpt.pt \
    --steps 1,2,4,8,16,50,100 --n-samples 5000 --cfg-scale 1.0 --metric both

In [ ]:
import json
fid = json.load(open(f'{REPO}/results/fid/fid.json'))
for metric, per_method in fid.items():
    print('==', metric, '==')
    for name, d in per_method.items():
        print(name, {int(k): round(v, 2) for k, v in d.items()})

In [ ]:
for metric in ('inception', 'mnist'):
    display(Image(f'{REPO}/results/fid/fid_vs_steps_{metric}.png'))

## 5. Sample grids: few vs. many steps

In [ ]:
for ck in ('flow_mnist', 'ddpm_mnist', 'reflow_mnist'):
    !cd {REPO} && PYTHONPATH={REPO} python -m scripts.sample --ckpt results/{ck}/ckpt.pt --steps 2,8,100 --per-class 10 --cfg-scale 2.0
    for s in (2, 8, 100):
        print(f'{ck}: {s} steps'); display(Image(f'{REPO}/results/{ck}/samples_steps{s}.png'))